# Week 6: Linear Regression & Model Interpretation

This notebook explores the core concepts of linear regression. We analyze real world datasets from Week 5 (IMF World Economic Outlook and World Happiness Report) to explore the relation between country wealth and happiness.

This notebook has **three parts**:

1. **Part 1: Real World Regression (Happiness vs. GDP per Capita)**: Clean and merge datasets by country, create exploratory scatter plots, and fit a best fit regression line through actual data.
2. **Part 2: Model Interpretation & Specifications**: Compare and interpret four standard regression specifications (Level to Level, Level to Log, Log to Level, and Log to Log) using a dedicated interpretation utility function.
3. **Part 3: Appendix: Ordinary Least Squares (OLS) Fundamentals (Optional)**: Learn how OLS is solved under the hood. Using our country dataset, we calculate regression coefficients by hand, compare them with statsmodels, visualize regression errors, and solve OLS using matrix algebra.
<hr>


# Part 1: Real World Regression (Happiness vs. GDP per Capita)

We start by loading and merging our real datasets from Week 5: the IMF World Economic Outlook database and the World Happiness Report.


## 1.1 Load and Merge Datasets

We load the datasets from `data/examples/week_5/`, clean the column names, and merge them for the year 2023.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load week 5 datasets
df_happy = pd.read_parquet("../../data/examples/week_5/world_happiness.parquet")
df_imf_raw = pd.read_parquet("../../data/examples/week_5/imf_weo_countries.parquet")

# Process IMF dataset: clean columns and filter for GDP per capita (NGDPDPC) in 2023
df_imf_raw.columns = df_imf_raw.columns.str.lower().str.replace(" ", "_")
df_gdp_2023 = df_imf_raw.query("weo_subject_code == 'NGDPDPC' and year == 2023")[['country', 'value']].dropna()
df_gdp_2023 = df_gdp_2023.rename(columns={'country': 'country_name', 'value': 'gdp_per_capita'})

# Merge datasets by country
df_merged = pd.merge(df_happy, df_gdp_2023, on='country_name')
df_merged = df_merged[['country_name', 'ladder_score', 'gdp_per_capita']].dropna()
print(f"Total merged records: {len(df_merged)}")
df_merged.head(5)


## 1.2 Exploratory Scatter Plot

We first inspect the relationship visually by creating a scatter plot.


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_merged['gdp_per_capita'], df_merged['ladder_score'], color='purple', alpha=0.7)
plt.title("Exploratory Scatter Plot: Happiness Score vs. GDP per Capita")
plt.xlabel("GDP per Capita (USD)")
plt.ylabel("Happiness Ladder Score")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


## 1.3 Fitting the Regression Line

We fit a simple linear regression model to the merged data and draw the best fit regression line directly through our scatter plot.


In [ ]:
import statsmodels.api as sm

# Prepare independent and dependent variables
X_real = df_merged['gdp_per_capita']
y_real = df_merged['ladder_score']
X_real_const = sm.add_constant(X_real)

# Fit OLS model
model_real = sm.OLS(y_real, X_real_const).fit()
intercept_real, slope_real = model_real.params.iloc[0], model_real.params.iloc[1]

# Plot data with regression line
plt.figure(figsize=(10, 6))
plt.scatter(df_merged['gdp_per_capita'], df_merged['ladder_score'], color='purple', alpha=0.7, label='Countries')
plt.plot(df_merged['gdp_per_capita'], intercept_real + slope_real * df_merged['gdp_per_capita'], color='red', label='OLS Best Fit Regression Line')
plt.title("Happiness Score vs. GDP per Capita with Regression Line")
plt.xlabel("GDP per Capita (USD)")
plt.ylabel("Happiness Ladder Score")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print(model_real.summary())


# Part 2: Model Interpretation & Specifications

Not all relationships are purely linear. In economics and public policy, variables are often transformed using logarithms to model percentage changes and elasticities. We will examine four specifications:

1. **Level to Level**: Standard linear relationship ($y = \alpha + \beta x$).
2. **Level to Log**: Independent variable is log transformed ($y = \alpha + \beta \ln(x)$).
3. **Log to Level**: Dependent variable is log transformed ($\ln(y) = \alpha + \beta x$).
4. **Log to Log**: Both variables are log transformed ($\ln(y) = \alpha + \beta \ln(x)$).


## 2.1 Creating Log Transformed Features

We apply natural log transforms to our target columns.


In [ ]:
# Add log transformed columns
df_merged['gdp_log'] = np.log(df_merged['gdp_per_capita'])
df_merged['ladder_log'] = np.log(df_merged['ladder_score'])
df_merged.head(2)


## 2.2 Defining the Regression Interpretation Utility

We define a helper function to run regressions and output the exact mathematical and economic interpretation of the coefficients without any OOP code.


In [ ]:
def regression_interp(data, y_col, x_col, y_type, x_type, y_name, x_name):
    y_series = data[y_col]
    x_series = data[x_col]
    x_with_const = sm.add_constant(x_series)
    model = sm.OLS(y_series, x_with_const).fit()
    
    coefficient_val = model.params.loc[x_col]
    r_squared_val = model.rsquared
    p_value_val = model.pvalues.loc[x_col]
    
    print(f"\n=== Regression Interpretation: {y_name} vs. {x_name} ===")
    print(f"Specification: {y_type.upper()} to {x_type.upper()}")
    print(f"Coefficient: {coefficient_val:.6f}")
    
    # Print interpretations based on specification type
    if y_type == 'log' and x_type == 'log':
        print(f"Interpretation: A 1% increase in {x_name} is associated with a {coefficient_val:.4f}% change in {y_name}.")
    elif y_type == 'level' and x_type == 'log':
        change_y = coefficient_val / 100
        print(f"Interpretation: A 1% increase in {x_name} is associated with a {change_y:.4f} unit change in {y_name}.")
    elif y_type == 'log' and x_type == 'level':
        percentage_y = coefficient_val * 100
        print(f"Interpretation: A 1 unit increase in {x_name} is associated with a {percentage_y:.4f}% change in {y_name}.")
    elif y_type == 'level' and x_type == 'level':
        print(f"Interpretation: A 1 unit increase in {x_name} is associated with a {coefficient_val:.6f} unit change in {y_name}.")
    
    print(f"R squared (Fit quality): {r_squared_val:.4f}")
    print(f"p value (Statistical significance): {p_value_val:.4f}")


## 2.3 Comparing the Four Specifications

We run the interpretation utility on all four combinations to see their coefficients, fit quality, and significance levels.


In [ ]:
# Run level to level
regression_interp(df_merged, 'ladder_score', 'gdp_per_capita', 'level', 'level', 'Happiness Score', 'GDP per capita')

# Run level to log
regression_interp(df_merged, 'ladder_score', 'gdp_log', 'level', 'log', 'Happiness Score', 'GDP per capita')

# Run log to level
regression_interp(df_merged, 'ladder_log', 'gdp_per_capita', 'log', 'level', 'Happiness Score', 'GDP per capita')

# Run log to log
regression_interp(df_merged, 'ladder_log', 'gdp_log', 'log', 'log', 'Happiness Score', 'GDP per capita')


## 2.4 Visualizing the Four Specifications Side by Side

We create a grid of plots to visualize how each model behaves. Notice how using the log of GDP per capita straightens out the curvature, showing a highly linear relationship with happiness!


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Level to Level
axes[0, 0].scatter(df_merged['gdp_per_capita'], df_merged['ladder_score'], color='purple', alpha=0.6)
m_ll = sm.OLS(df_merged['ladder_score'], sm.add_constant(df_merged['gdp_per_capita'])).fit()
axes[0, 0].plot(df_merged['gdp_per_capita'], m_ll.params.iloc[0] + m_ll.params.iloc[1] * df_merged['gdp_per_capita'], color='red')
axes[0, 0].set_title("Level to Level (Linear Relation)")
axes[0, 0].set_xlabel("GDP per capita")
axes[0, 0].set_ylabel("Happiness Score")

# 2. Level to Log
axes[0, 1].scatter(df_merged['gdp_log'], df_merged['ladder_score'], color='blue', alpha=0.6)
m_l_log = sm.OLS(df_merged['ladder_score'], sm.add_constant(df_merged['gdp_log'])).fit()
sort_gdp = df_merged.sort_values('gdp_log')
axes[0, 1].plot(sort_gdp['gdp_log'], m_l_log.params.iloc[0] + m_l_log.params.iloc[1] * sort_gdp['gdp_log'], color='red')
axes[0, 1].set_title("Level to Log (Linear in Logs)")
axes[0, 1].set_xlabel("Log of GDP per capita")
axes[0, 1].set_ylabel("Happiness Score")

# 3. Log to Level
axes[1, 0].scatter(df_merged['gdp_per_capita'], df_merged['ladder_log'], color='green', alpha=0.6)
m_log_l = sm.OLS(df_merged['ladder_log'], sm.add_constant(df_merged['gdp_per_capita'])).fit()
sort_gdp_pc = df_merged.sort_values('gdp_per_capita')
axes[1, 0].plot(sort_gdp_pc['gdp_per_capita'], m_log_l.params.iloc[0] + m_log_l.params.iloc[1] * sort_gdp_pc['gdp_per_capita'], color='red')
axes[1, 0].set_title("Log to Level")
axes[1, 0].set_xlabel("GDP per capita")
axes[1, 0].set_ylabel("Log of Happiness Score")

# 4. Log to Log
axes[1, 1].scatter(df_merged['gdp_log'], df_merged['ladder_log'], color='orange', alpha=0.6)
m_log_log = sm.OLS(df_merged['ladder_log'], sm.add_constant(df_merged['gdp_log'])).fit()
sort_gdp_l = df_merged.sort_values('gdp_log')
axes[1, 1].plot(sort_gdp_l['gdp_log'], m_log_log.params.iloc[0] + m_log_log.params.iloc[1] * sort_gdp_l['gdp_log'], color='red')
axes[1, 1].set_title("Log to Log (Constant Elasticity Relation)")
axes[1, 1].set_xlabel("Log of GDP per capita")
axes[1, 1].set_ylabel("Log of Happiness Score")

plt.tight_layout()
plt.show()


# Part 3: Appendix: Ordinary Least Squares (OLS) Fundamentals (Optional)

In this optional section, we explore the mathematics of OLS from first principles using our actual merged country dataset (Happiness vs. GDP per capita).


## 3.1 Calculating OLS Coefficients by Hand

Using our real data, we calculate the covariance and variance of `gdp_per_capita` and `ladder_score` to find the slope (beta) and intercept (alpha) coefficients manually:

$$\hat\beta = \frac{Cov(x, y)}{Var(x)}$$
$$\hat\alpha = \bar{y} - \hat\beta \bar{x}$$


In [ ]:
# Calculate means of our real variables
x_mean = df_merged['gdp_per_capita'].mean()
y_mean = df_merged['ladder_score'].mean()

# Calculate differences from the mean
df_merged['xt'] = df_merged['gdp_per_capita'] - x_mean
df_merged['yt'] = df_merged['ladder_score'] - y_mean
df_merged['xt2'] = df_merged['xt'] ** 2

# Calculate slope (beta) and intercept (alpha) by hand
beta_hand = sum(df_merged['xt'] * df_merged['yt']) / sum(df_merged['xt2'])
alpha_hand = y_mean - beta_hand * x_mean

print("Calculated by hand on real country data:")
print(f"Beta (slope) = {beta_hand:.8f}")
print(f"Alpha (intercept) = {alpha_hand:.4f}")


## 3.2 Verifying with Statsmodels

We verify that our manual coefficients match statsmodels exactly.


In [ ]:
X_append = df_merged['gdp_per_capita']
X_append_const = sm.add_constant(X_append)
y_append = df_merged['ladder_score']

ols_append = sm.OLS(y_append, X_append_const).fit()
beta_model = ols_append.params.iloc[1]
alpha_model = ols_append.params.iloc[0]

print("Calculated using Statsmodels:")
print(f"Beta (slope) = {beta_model:.8f}")
print(f"Alpha (intercept) = {alpha_model:.4f}")


## 3.3 Visualizing OLS Errors (TSS, ESS, RSS) on Real Data

We visualize the Total Sum of Squares (TSS), Explained Sum of Squares (ESS), and Residual Sum of Squares (RSS).

To keep the visualization clean and readable, we display the error arrows for a representative sample of 15 countries. We scale the arrow widths to fit the scale of the GDP per capita axis.


In [ ]:
# Calculate predicted values, residuals, and mean for the full dataset
df_merged['y_hat'] = alpha_hand + df_merged['gdp_per_capita'] * beta_hand
df_merged['u_hat'] = df_merged['ladder_score'] - df_merged['y_hat']
df_merged['y_mean'] = y_mean

# Sum of Squares
ess = sum((df_merged['y_hat'] - y_mean) ** 2)
rss = sum((df_merged['ladder_score'] - df_merged['y_hat']) ** 2)
tss = sum((df_merged['ladder_score'] - y_mean) ** 2)
r2 = 1 - rss / tss

# Set up the background plot with all countries
plt.figure(figsize=(12, 7))
plt.scatter(df_merged['gdp_per_capita'], df_merged['ladder_score'], color='purple', alpha=0.3, label='All Countries')
plt.plot(df_merged['gdp_per_capita'], df_merged['y_hat'], color='red', label='Best Fit OLS Regression Line')
plt.axhline(y_mean, color='green', linestyle=':', label='Mean of Happiness Score')

# Draw arrows for a sample of 15 countries to keep the graph legible
np.random.seed(42)
df_sample = df_merged.sample(15)
x_sample = df_sample['gdp_per_capita'].values
y_sample = df_sample['ladder_score'].values
y_hat_sample = df_sample['y_hat'].values

for i in range(len(df_sample)):
    # TSS arrow (pink) - representing distance from the mean of Y
    plt.arrow(x_sample[i], y_sample[i], 0, y_mean - y_sample[i], color='pink', alpha=0.6, width=150, head_width=450)
    # RSS arrow (lightblue) - representing distance from the regression line
    plt.arrow(x_sample[i], y_sample[i], 0, y_hat_sample[i] - y_sample[i], color='lightblue', alpha=0.6, width=150, head_width=450)
    # ESS arrow (orange) - representing distance between the regression line and mean of Y
    plt.arrow(x_sample[i], y_hat_sample[i], 0, y_mean - y_hat_sample[i], color='orange', alpha=0.6, width=150, head_width=450)

# Custom text box for legends
plt.text(df_merged['gdp_per_capita'].min(), df_merged['ladder_score'].max() * 0.95, 
         "TSS: Pink Arrows\nRSS: Light Blue Arrows\nESS: Orange Arrows", 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.title(f"Appendix: OLS Errors on Sample Countries (R squared = {r2:.3f})")
plt.xlabel("GDP per Capita (USD)")
plt.ylabel("Happiness Ladder Score")
plt.legend()
plt.show()


## 3.4 Solving OLS using Matrix Algebra

We solve for the coefficient vector $\hat{\beta}$ directly using matrix multiplication:

$$\hat{\beta} = (X^T X)^{-1} X^T Y$$


In [ ]:
# Convert our country variables to numpy matrices
n_obs = len(df_merged)
X_matrix = np.hstack((np.ones((n_obs, 1)), df_merged['gdp_per_capita'].values.reshape(-1, 1)))
Y_vector = df_merged['ladder_score'].values.reshape(-1, 1)

# Solve coefficients using matrix multiplication
beta_matrix = np.linalg.inv(X_matrix.T @ X_matrix) @ (X_matrix.T @ Y_vector)
intercept_matrix = beta_matrix[0][0]
slope_matrix = beta_matrix[1][0]

print("Calculated using Matrix Algebra on country data:")
print(f"Intercept = {intercept_matrix:.4f}")
print(f"Slope (GDP per capita) = {slope_matrix:.8f}")
print("\nCalculated using Statsmodels:")
print(f"Intercept = {ols_append.params.iloc[0]:.4f}")
print(f"Slope (GDP per capita) = {ols_append.params.iloc[1]:.8f}")
